# 🌡️ Why Is Your Neighborhood Hotter Than Mine?

**A space-based thermometer reveals the hidden geography of urban heat in Mexico City.**

*Notebook 1 of 8 · Project: Dos Méxicos Bajo el Mismo Sol*  
*Author: Nelly Itzel Rodríguez Ortiz · Last updated: June 2026*

## Some neighborhoods in Mexico City are up to 8 °C hotter than others. Why?

Take a walk through *Coyoacán* on a March afternoon: tree-lined streets, cool shade, a gentle breeze. Now take the same walk in *Iztapalapa*: the pavement radiates heat, bus stops feel like ovens, and the night barely cools down.

This isn't the weather. It's the **Urban Heat Island** effect, and it doesn't hit every neighborhood equally. In this notebook we'll use a satellite "space thermometer" to map the surface temperature of every corner of Mexico City and find out **who lives in the hottest places**.

## Why this matters

Heat is more than just discomfort — it is a **public-health and environmental-justice question** that Mexico City is only beginning to grapple with.

- 🌡️ **Heat is on the rise.** In May 2025, Mexico City recorded its hottest May on record.
- 🏘️ **The city is not uniform.** Some neighborhoods are far hotter than others, and the reasons are **not yet clear**.
- ❓ **This notebook is just step one.** It maps the temperature. The *why* — and the *what to do about it* — comes later in the series.

> **In one line:** *Mexico City's summer is not the same for everyone. Let's find out by how much.*

## 🛰️ What is "Land Surface Temperature" (LST)?

**Land Surface Temperature (LST)** is exactly what it sounds like: how hot the ground is. But instead of putting a thermometer in the dirt, we measure it **from space**.

NASA's **Landsat 8 and 9 satellites** orbit Earth about **700 km up** and photograph every spot on the planet every 16 days. One of their instruments is sensitive to **infrared light** — the same invisible warmth your hand gives off. By measuring that glow, the satellite can calculate the temperature of rooftops, streets, parks, and reservoirs with remarkable precision.

> **Why use a satellite?** A weather station tells you about *one point*; a satellite tells you about *every square meter*, with the same instrument, on the same day. That makes it the only fair way to compare heat across an entire city.

> **The catch:** LST measures the *ground*, not the *air*. On a sunny day, an asphalt parking lot can be **25 °C hotter than the air** above it. LST is closer to what your *feet* feel than to what your weather app reports.

## 🗂️ Where the data comes from

| Source | What we get | Window |
|---|---|---|
| **Landsat 8 / 9** (NASA–USGS) | Cloud-free surface temperature composites | Summer 2025 (Jun–Aug) and Winter 2025–2026 (Dec–Feb) |
| **INEGI Marco Geoestadístico (municipal)** | Boundary of the ZMVM (16 CDMX boroughs + 60 EdoMex municipios) | 2020 |

All the heavy lifting — cloud masking, unit conversion, image compositing — is done by the `src/` Python package. This notebook focuses on **telling the story**.

### ⚙️ One-time setup
The first time you run this notebook, uncomment the `ee.Authenticate()` line and follow the browser prompt. After that, the kernel remembers you for the session.

In [6]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so `import src` works from any CWD

import ee
import geemap

from src import (
    CDMX_CENTER, EE_PROJECT_ID, LST_VIS_PARAMS,
    get_zmvm_municipalities_aoi,
    load_lst_composite,
)

ee.Authenticate()  # ← uncomment the first time
ee.Initialize(project=EE_PROJECT_ID)
print("✅ Earth Engine ready")


/snap/core20/current/lib/x86_64-linux-gnu/libstdc++.so.6: version `GLIBCXX_3.4.29' not found (required by /lib/x86_64-linux-gnu/libproxy.so.1)
Failed to load module: /home/nells-it/snap/code/common/.cache/gio-modules/libgiolibproxy.so


Abriendo en una sesión existente del navegador

Successfully saved authorization token.
✅ Earth Engine ready


## 🌡️ Building the map, step by step

A single Landsat snapshot has two big problems: **clouds** (which look cold) and **gaps** (the satellite only passes every 16 days). To get a clean view of the city we follow four steps:

1. **Collect** every clear-sky Landsat image of the ZMVM for the season.
2. **Mask out the clouds** using the satellite's own quality flags.
3. **Convert** the raw sensor units to degrees Celsius.
4. **Take the median** of all the clean images. The median ignores the occasional weird reading (a stray cloud, a sensor glitch) the way the **house price in the middle of the street** ignores the one mansion that would skew an average.

The result: a single, clean temperature map for the whole metropolitan area.

In [7]:
# Build the two seasonal composites. Everything else (cloud mask, K→°C,
# median) lives inside load_lst_composite in src/landsat.py.

aoi = get_zmvm_municipalities_aoi()                       # ZMVM: 16 CDMX + 60 EdoMex municipios

lst_summer = load_lst_composite(
    aoi, start_date="2025-06-01", end_date="2025-09-01", cloud_max=20.0,
)
lst_winter = load_lst_composite(
    aoi, start_date="2025-12-01", end_date="2026-03-01", cloud_max=20.0,
)

print("Summer composite — band:", lst_summer.bandNames().getInfo())
print("Winter composite — band:", lst_winter.bandNames().getInfo())


Summer composite — band: ['LST_C']
Winter composite — band: ['LST_C']


### 🗺️ Map: Mexico City's surface temperature, summer vs winter

Use the **layer control** (top right) to toggle between seasons. Hover over the map to read the temperature at any point. **Cool blues are the lowest; deep reds are the hottest.**

In [8]:
Map = geemap.Map(center=CDMX_CENTER, zoom=11)

Map.addLayer(lst_summer, LST_VIS_PARAMS, "🌡️ LST — Summer 2025")
Map.addLayer(lst_winter, LST_VIS_PARAMS, "❄️ LST — Winter 2025–2026 (Dec–Feb)")
Map.addLayer(
    ee.Image().paint(aoi, 0, 2), {"palette": ["white"]}, "ZMVM boundary",
)
Map.add_colorbar(
    LST_VIS_PARAMS,
    label="Surface temperature (°C)",
    layer_name="🌡️ LST — Summer 2025",
)
Map.add_layer_control()
Map


Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topr…

> 🔍 **What to look for:** Even at this zoom, the pattern jumps out. The **north and east of the metropolitan area glow red**, while the **southern mountains and conserved areas (Desierto de los Leones, the Pedregal) stay cool and blue.** The signal is *stronger in summer* but **does not disappear in winter** — a clue that the gap is built into the city itself, not just a seasonal accident.

## ⚖️ A tale of two Mexicos: north vs south

Before jumping to conclusions about *why* the north might be hotter, let's first confirm *how much* hotter it really is — using actual measurements, not intuition.

We compare the **northeast** of the ZMVM (Gustavo A. Madero, Iztapalapa, Ecatepec de Morelos, Tlalnepantla de Baz, and Nezahualcóyotl) with the **south** (Coyoacán, Álvaro Obregón, Tlalpan, La Magdalena Contreras). We select municipios by name for now; in Notebook 4 we'll do this properly with the official AGEB layer from INEGI.

In [9]:
# Pick the northeast and south municipios from the ZMVM FeatureCollection.

nororiente_names = [
    "Gustavo A. Madero", "Iztapalapa", "Ecatepec de Morelos",
    "Tlalnepantla de Baz", "Nezahualcóyotl",
]
sur_names = [
    "Coyoacán", "Álvaro Obregón", "Tlalpan", "La Magdalena Contreras",
]

nororiente = aoi.filter(ee.Filter.inList("NOMGEO", nororiente_names))
sur        = aoi.filter(ee.Filter.inList("NOMGEO", sur_names))

def mean_lst(image, fc):
    return image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=fc.geometry().dissolve(),
        scale=30, maxPixels=1e9,
    ).getInfo()["LST_C"]

m_n_s, m_s_s = mean_lst(lst_summer, nororiente), mean_lst(lst_summer, sur)
m_n_w, m_s_w = mean_lst(lst_winter, nororiente), mean_lst(lst_winter, sur)

print(f"Summer  — North: {m_n_s:.1f}°C  |  South: {m_s_s:.1f}°C  |  Gap: {m_n_s - m_s_s:+.1f}°C")
print(f"Winter  — North: {m_n_w:.1f}°C  |  South: {m_s_w:.1f}°C  |  Gap: {m_n_w - m_s_w:+.1f}°C")


Summer  — North: 36.6°C  |  South: 26.0°C  |  Gap: +10.6°C
Winter  — North: 28.5°C  |  South: 21.6°C  |  Gap: +6.8°C


> 🔑 **Key finding:** The data is clear: a **5–10 °C gap** exists between the northeast and the south of the ZMVM — in both summer and winter. The gap does not close in December.
>
> But what drives it? Our first suspect: **vegetation**. Or rather, the **lack** of it. That's exactly what we'll measure next.

## 🫁 What COULD a 5 °C gap mean?

A 5 °C difference on a thermometer might sound small. In the body, it isn't. Here is what **public-health research around the world** has linked to temperature gaps of this size — *as correlations, not certainties for CDMX*:

| Gap | Potential impact based on public health research |
|---|---|
| **+1 °C** | Heat-wave thresholds may be crossed; advisories are sometimes issued to older adults |
| **+3 °C** | Risk of heat exhaustion can rise sharply for outdoor workers (vendors, construction, transit) |
| **+5 °C** | Pediatric asthma ER visits have been **observed to double** in some studies on the hottest days |
| **+8 °C** | The kind of gap seen between ZMVM's north and south during a heat wave — **may be associated with elevated risk for infants and older adults without cooling access** |

> **Important caveat:** These are correlations observed globally, not measurements of CDMX specifically. Whether they hold true here — and to what degree — is what we will investigate when we analyze local health data in **Notebook 6**.

---

## 🔍 What we know, and what we don't

✅ **Confirmed in this notebook:** The northern and eastern parts of the Mexico City metro area are **5–8 °C hotter** than the south — in both summer and winter.

❓ **Still open questions:**

- **Is this gap explained by lack of vegetation?** → *Notebook 2: NDVI (next)*
- **Is the hotter north also the more polluted north?** → *Notebook 3: NO₂ & PM₂.₅*
- **Do poorer neighborhoods suffer more from heat?** → *Notebook 5: Marginalization*
- **Does the heat make people sick?** → *Notebook 6: Respiratory health*
- **Do residents themselves perceive this inequality?** → *Notebook 7: Perception survey*

📌 **The final answer — and what we should do about it — will come in Notebook 8**, once we have all the evidence side by side.

---
### 📚 Learn more
- NASA Landsat 8 mission — https://landsat.gsfc.nasa.gov/landsat-8-9/
- WMO & WHO — *Heat and Health* (2023)
- World Bank — *Climate Risk and Opportunity in Mexico City* (2022)
- INEGI — *Censo de Población y Vivienda 2020*

### 🛠️ About the code
- All reusable helpers live in the `src/` package.
- This notebook is a thin **orchestrator** — it imports helpers and stitches the story together.
- Reproducibility: pinned versions in `requirements.txt`; the `src/` package is installed in editable mode.

---

➡️ **Next notebook:** *Where Are the Trees? — Mapping Vegetation (NDVI) Across CDMX*